# ResNet-50 thresholds for Macaroni 1 and PCB 2
Load the selected fine-tuned model and inspect these two products. Choose thresholds using the same validation images held out from this model's training. Compare with its original shared threshold on test images afterward.

Run top to bottom on BigMac using dl_gpu. This notebook performs inference only. The saved model and VisA images must already exist locally. Each product has only 12 validation defects, so a threshold may perform differently on test images. Previous test inspection makes this an exploratory experiment.


## 1. Settings and saved model

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet50_Weights, resnet50

PRODUCTS = ["macaroni1", "pcb2"]
TARGET_RECALL = 0.90
BATCH_SIZE = 16
DATA_DIR = Path("visa_data")
MODEL_DIR = Path("results/full_visa_resnet50_parallel")
OUTPUT_DIR = Path("results/macaroni1_pcb2_thresholds")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)

selected = pd.read_csv(MODEL_DIR / "selected_run.csv", index_col=0)["value"]
original_threshold = float(selected["threshold"])
checkpoint = Path(selected["checkpoint"])
assert checkpoint.is_file(), f"Missing saved model: {checkpoint}"
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
model = model.to(device).eval()
preprocess = ResNet50_Weights.DEFAULT.transforms()
print("Using", device, "Original threshold", original_threshold)


## 2. Recreate the validation split
Split the complete training list first, exactly as in the training notebook. Then select the two products.

In [ ]:
rows = pd.read_csv(DATA_DIR / "2cls_highshot.csv")
training = rows[rows["split"] == "train"].reset_index(drop=True)
strata = training["object"] + "_" + training["label"]
_, validation_indices = train_test_split(
    range(len(training)), test_size=0.2, stratify=strata,
    random_state=int(selected["split_seed"]),
)
validation = training.iloc[validation_indices]
validation = validation[validation["object"].isin(PRODUCTS)].reset_index(drop=True)
testing = rows[(rows["split"] == "test") & rows["object"].isin(PRODUCTS)].reset_index(drop=True)
assert set(validation["image"]).isdisjoint(testing["image"])
display(pd.crosstab(validation["object"], validation["label"]))

class ImageDataset(Dataset):
    def __init__(self, table):
        self.table = table.reset_index(drop=True)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        path = DATA_DIR / self.table.iloc[index]["image"]
        with Image.open(path) as image:
            return preprocess(image.convert("RGB"))

@torch.no_grad()
def predict(table):
    probabilities = []
    for images in DataLoader(ImageDataset(table), batch_size=BATCH_SIZE, shuffle=False):
        scores = model(images.to(device)).squeeze(1).sigmoid()
        probabilities.extend(scores.cpu().tolist())
    result = table[["object", "image", "label"]].copy()
    result["label"] = result["label"].map({"normal": 0, "anomaly": 1})
    result["score"] = probabilities
    return result

validation_scores = predict(validation)


## 3. Inspect validation scores and false alarms

In [ ]:
for product in PRODUCTS:
    group = validation_scores[validation_scores["object"] == product]
    plt.figure(figsize=(6, 3))
    for label, name in [(0, "Normal"), (1, "Defective")]:
        plt.hist(group.loc[group["label"] == label, "score"], bins=20, range=(0, 1), alpha=0.5, label=name)
    plt.axvline(original_threshold, color="black", linestyle="--", label="Original cutoff")
    plt.title(product)
    plt.xlabel("Defect score")
    plt.ylabel("Images")
    plt.legend()
    plt.show()

    false_alarms = group[(group["label"] == 0) & (group["score"] >= original_threshold)]
    for row in false_alarms.head(3).itertuples():
        with Image.open(DATA_DIR / row.image) as image:
            plt.figure(figsize=(3, 3))
            plt.imshow(image)
            plt.title(f"{product}: normal, score={row.score:.3f}")
            plt.axis("off")
            plt.show()


## 4. Compare validation thresholds
For each product, choose the highest cutoff that reaches 90% validation recall. Show it alongside a short list of fixed cutoffs and the original shared cutoff. Each selected threshold must detect at least 11 of the 12 validation defects.

In [ ]:
def summarize(group, threshold):
    predicted = group["score"] >= threshold
    tn, fp, fn, tp = confusion_matrix(group["label"], predicted, labels=[0, 1]).ravel()
    return {
        "threshold": threshold, "detected": tp, "missed": fn, "false_alarms": fp,
        "recall": tp / (tp + fn),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
    }

thresholds = {}
comparison_rows = []
for product in PRODUCTS:
    group = validation_scores[validation_scores["object"] == product]
    defect_scores = group.loc[group["label"] == 1, "score"].sort_values(ascending=False)
    required = math.ceil(TARGET_RECALL * len(defect_scores))
    thresholds[product] = float(defect_scores.iloc[required - 1])
    cutoffs = sorted(set([original_threshold, 0.01, 0.05, 0.1, 0.3, 0.5, thresholds[product]]))
    for cutoff in cutoffs:
        comparison_rows.append({
            "product": product, "selected": cutoff == thresholds[product],
            **summarize(group, cutoff),
        })
validation_comparison = pd.DataFrame(comparison_rows)
display(validation_comparison.round(4))


## 5. Compare test results and save
Apply the chosen thresholds once. Compare both approaches on exactly the same images. Positive changes in missed defects mean more defects were missed. Negative changes in false alarms mean fewer false alarms.

In [ ]:
test_scores = predict(testing)
result_rows = []
for product in PRODUCTS:
    group = test_scores[test_scores["object"] == product]
    for method, cutoff in [("Original", original_threshold), ("Product threshold", thresholds[product])]:
        result_rows.append({"product": product, "method": method, **summarize(group, cutoff)})
results = pd.DataFrame(result_rows)
display(results.round(4))

for product in PRODUCTS:
    group = results[results["product"] == product].set_index("method")
    change = group.loc["Product threshold", ["false_alarms", "missed"]] - group.loc["Original", ["false_alarms", "missed"]]
    print(product, "Change in false alarms:", int(change["false_alarms"]),
          "Change in missed defects:", int(change["missed"]))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_scores.to_csv(OUTPUT_DIR / "validation_scores.csv", index=False)
validation_comparison.to_csv(OUTPUT_DIR / "validation_thresholds.csv", index=False)
test_scores.to_csv(OUTPUT_DIR / "test_scores.csv", index=False)
results.to_csv(OUTPUT_DIR / "test_comparison.csv", index=False)
print("Saved results to", OUTPUT_DIR.resolve())
